In [1]:
# imports

from dotenv import load_dotenv
from agents import (
    Agent, Runner, trace, function_tool,
    WebSearchTool, ModelSettings,
    input_guardrail, GuardrailFunctionOutput,
)
from pydantic import BaseModel, Field
import asyncio
import os
import requests
import gradio as gr

In [2]:
# Constants

MODEL = "gpt-4o-mini"
HOW_MANY_SEARCHES = 3

In [3]:
# Environment setup 

load_dotenv(override=True)

openai_api_key    = os.getenv("OPENAI_API_KEY")
pushover_user     = os.getenv("PUSHOVER_USER")
pushover_token    = os.getenv("PUSHOVER_TOKEN")

print("OpenAI API Key:", "✅" if openai_api_key  else "❌ not set")
print("Pushover user: ", "✅" if pushover_user   else "❌ not set")
print("Pushover token:", "✅" if pushover_token  else "❌ not set")

OpenAI API Key: ✅
Pushover user:  ✅
Pushover token: ✅


In [4]:
# Pydantic schemas

class TopicCheck(BaseModel):
    is_allowed: bool = Field(description="True if the query is about agriculture, commodities, food markets, or related finance/economics.")
    reason: str      = Field(description="Brief explanation of why the query was allowed or blocked.")

class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search is important to the query.")
    query:  str = Field(description="The search term to use.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="List of web searches to perform.")

class ReportData(BaseModel):
    short_summary:       str        = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report:     str        = Field(description="The full report in markdown format.")
    follow_up_questions: list[str]  = Field(description="Suggested topics to research further.")



In [5]:
# Guardrail agent + guardrail function 

topic_guard_agent = Agent(
    name="TopicGuardAgent",
    instructions=(
        "You decide whether a research query is on-topic for an agricultural and commodity markets "
        "research tool. Allowed topics include: crop prices, food commodities, livestock, fertiliser, "
        "agri-finance, commodity exchanges, supply chains, food security, weather impact on harvests, "
        "and macroeconomic topics directly linked to agriculture or commodity markets. "
        "Block anything clearly unrelated -- entertainment, sports, general coding help, medical advice, etc. "
        "When in doubt, allow it."
    ),
    output_type=TopicCheck,
    model=MODEL,
)

@input_guardrail
async def agri_topic_guardrail(ctx, agent, message):
    result = await Runner.run(topic_guard_agent, message, context=ctx.context)
    check  = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason},
        tripwire_triggered=not check.is_allowed,
    )


In [6]:
# Planner agent

planner_agent = Agent(
    name="PlannerAgent",
    instructions=(
        f"You are a helpful research assistant specialising in agriculture and commodity markets. "
        f"Given a query, produce exactly {HOW_MANY_SEARCHES} targeted web search terms that will "
        f"collectively give a thorough picture of the topic."
    ),
    output_type=WebSearchPlan,
    model=MODEL,
    input_guardrails=[agri_topic_guardrail],
)


In [7]:
# Search agent

search_agent = Agent(
    name="SearchAgent",
    instructions=(
        "You are a research assistant. Given a search term, search the web and produce a concise "
        "summary of the results in 2-3 paragraphs, under 300 words. Capture the main points. "
        "Write succinctly -- this will be consumed by someone synthesising a report. "
        "Output only the summary, no additional commentary."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)


In [8]:
# Writer agent

writer_agent = Agent(
    name="WriterAgent",
    instructions=(
        "You are a senior researcher specialising in agricultural and commodity markets. "
        "You will be given the original query and summarised search results. "
        "First outline the report structure, then write the full report. "
        "Output must be detailed markdown, at least 1000 words, covering key findings, "
        "market context, regional impacts where relevant, and implications."
    ),
    output_type=ReportData,
    model=MODEL,
)

In [9]:
# Pushover notification tool 

@function_tool
def send_push_notification(message: str) -> str:
    """Send a push notification via Pushover to alert the user that research is complete."""
    try:
        response = requests.post(
            "https://api.pushover.net/1/messages.json",
            data={
                "token":   pushover_token,
                "user":    pushover_user,
                "message": message,
                "title":   "Agri Research Complete",
            },
        )
        response.raise_for_status()
        return "Push notification sent."
    except Exception as e:
        return f"Push notification failed: {e}"

In [10]:
# Notifier agent

notifier_agent = Agent(
    name="NotifierAgent",
    instructions=(
        "You receive a completed research report. "
        "Use the send_push_notification tool to send the user a concise summary "
        "(2-3 sentences max) and let them know the full report is ready in the app. "
        "Call the tool exactly once."
    ),
    tools=[send_push_notification],
    model=MODEL,
)


In [11]:
# Research pipeline

async def plan_searches(query: str) -> WebSearchPlan:
    result = await Runner.run(planner_agent, f"Query: {query}")
    return result.final_output_as(WebSearchPlan)

async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    tasks   = [asyncio.create_task(_search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    return [r for r in results if r]

async def _search(item: WebSearchItem) -> str | None:
    try:
        result = await Runner.run(
            search_agent,
            f"Search term: {item.query}\nReason: {item.reason}",
        )
        return str(result.final_output)
    except Exception:
        return None

async def write_report(query: str, search_results: list[str]) -> ReportData:
    result = await Runner.run(
        writer_agent,
        f"Original query: {query}\nSearch summaries: {search_results}",
    )
    return result.final_output_as(ReportData)

async def notify(report: ReportData) -> None:
    await Runner.run(notifier_agent, report.markdown_report)


In [12]:
# Gradio generator function

async def run_research(query: str):
    yield "Planning searches..."
    try:
        with trace("Agri Research"):
            search_plan = await plan_searches(query)
            yield f"Search plan ready -- {len(search_plan.searches)} searches queued."

            search_results = await perform_searches(search_plan)
            yield f"Searches complete ({len(search_results)} results). Writing report..."

            report = await write_report(query, search_results)
            yield "Report written. Sending push notification..."

            await notify(report)
            yield "Push notification sent.\n\n---\n\n" + report.markdown_report

    except Exception as e:
        err = str(e)
        if "tripwire" in err.lower() or "guardrail" in err.lower():
            yield (
                "**Query blocked.** This tool covers agriculture, commodities, food markets, "
                "and related finance only. Please rephrase your query or try a different topic."
            )
        else:
            yield f"**Error:** {err}"

In [13]:
# Gradio UI

with gr.Blocks() as ui:
    gr.Markdown(
        "# Agri-Commodity Research Agent\n"
        "Deep research on agriculture, food markets, and commodity prices. "
        "Enter a query below -- the agent will plan, search, and synthesise a full report, "
        "then push a summary to your phone."
    )
    query_box  = gr.Textbox(label="Research query", placeholder="e.g. Impact of El Nino on West African cocoa prices 2024")
    submit_btn = gr.Button("Research", variant="primary")
    output     = gr.Markdown(label="Report")

    submit_btn.click(fn=lambda: gr.update(interactive=False), outputs=submit_btn)\
              .then(fn=run_research, inputs=query_box, outputs=output)\
              .then(fn=lambda: gr.update(interactive=True), outputs=submit_btn)

    query_box.submit(fn=lambda: gr.update(interactive=False), outputs=submit_btn)\
             .then(fn=run_research, inputs=query_box, outputs=output)\
             .then(fn=lambda: gr.update(interactive=True), outputs=submit_btn)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
